# Problem Statement 7: Text Analytics
**Objective:**
1. Apply NLP preprocessing: Tokenization, POS Tagging, Stop Words Removal, Stemming, Lemmatization.
2. Compute Term Frequency (TF) and Inverse Document Frequency (IDF).

In [ ]:
import nltk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import math

# Download required NLTK data
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk import pos_tag
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

print("All libraries and NLTK data loaded!")

## Step 1: Sample Document

In [ ]:
document = """
Machine learning is a method of data analysis that automates analytical model building. 
It is based on the idea that systems can learn from data, identify patterns and make decisions with minimal human intervention. 
Deep learning is part of a broader family of machine learning methods based on artificial neural networks with representation learning.
Natural language processing is a subfield of linguistics, computer science, and artificial intelligence.
"""

print("Sample Document:")
print(document)

## Step 2: Tokenization
Tokenization splits text into individual words (word tokens) or sentences.

In [ ]:
# Sentence Tokenization
sentences = sent_tokenize(document)
print(f"Sentence Tokens ({len(sentences)} sentences):")
for i, s in enumerate(sentences, 1):
    print(f"  {i}. {s.strip()}")

In [ ]:
# Word Tokenization
word_tokens = word_tokenize(document)
print(f"Word Tokens ({len(word_tokens)} tokens):")
print(word_tokens[:20], '...')

## Step 3: POS Tagging (Part-of-Speech)
Assigns grammatical tags: NN=noun, VB=verb, JJ=adjective, RB=adverb, etc.

In [ ]:
pos_tags = pos_tag(word_tokens)
print("POS Tags (first 20):")
print(pos_tags[:20])

# Display as a table
pos_df = pd.DataFrame(pos_tags[:20], columns=['Word', 'POS Tag'])
print("\nPOS Tag Table:")
print(pos_df.to_string(index=False))

## Step 4: Stop Words Removal
Stop words are common words (e.g., 'is', 'the', 'a') that carry little meaningful information.

In [ ]:
stop_words = set(stopwords.words('english'))

# Keep only alphabetic tokens and remove stop words
filtered_tokens = [w.lower() for w in word_tokens if w.isalpha() and w.lower() not in stop_words]

print(f"Original token count: {len(word_tokens)}")
print(f"After stop word removal: {len(filtered_tokens)}")
print(f"\nFiltered Tokens: {filtered_tokens[:20]} ...")

## Step 5: Stemming
Stemming reduces words to their root form (may not always be a real word).
Example: 'learning' → 'learn', 'building' → 'build'

In [ ]:
stemmer = PorterStemmer()
stemmed_tokens = [stemmer.stem(word) for word in filtered_tokens]

print("Stemmed Tokens:")
stem_comparison = pd.DataFrame({'Original': filtered_tokens[:15], 'Stemmed': stemmed_tokens[:15]})
print(stem_comparison.to_string(index=False))

## Step 6: Lemmatization
Lemmatization reduces words to their base dictionary form (lemma) — always a valid word.
Example: 'running' → 'run', 'better' → 'good'

In [ ]:
lemmatizer = WordNetLemmatizer()
lemmatized_tokens = [lemmatizer.lemmatize(word) for word in filtered_tokens]

print("Lemmatized Tokens:")
lem_comparison = pd.DataFrame({
    'Original': filtered_tokens[:15],
    'Stemmed': stemmed_tokens[:15],
    'Lemmatized': lemmatized_tokens[:15]
})
print(lem_comparison.to_string(index=False))

## Step 7: TF-IDF (Term Frequency – Inverse Document Frequency)
**TF** = (Number of times term appears in doc) / (Total words in doc)

**IDF** = log(Total docs / Number of docs containing the term)

**TF-IDF** = TF × IDF

In [ ]:
# Corpus of documents
corpus = [
    "machine learning is used in data science",
    "deep learning is part of machine learning",
    "natural language processing uses machine learning",
    "data science requires statistical analysis and machine learning"
]

def compute_tf(doc):
    words = doc.lower().split()
    count = Counter(words)
    return {word: count[word] / len(words) for word in count}

def compute_idf(corpus):
    N = len(corpus)
    all_words = set(word for doc in corpus for word in doc.lower().split())
    idf = {}
    for word in all_words:
        containing = sum(1 for doc in corpus if word in doc.lower().split())
        idf[word] = math.log(N / (1 + containing))  # smoothed
    return idf

idf = compute_idf(corpus)

print("TF-IDF for each document:")
all_tfidf = []
for i, doc in enumerate(corpus):
    tf = compute_tf(doc)
    tfidf = {word: round(tf[word] * idf[word], 4) for word in tf}
    tfidf_sorted = dict(sorted(tfidf.items(), key=lambda x: x[1], reverse=True))
    all_tfidf.append(tfidf_sorted)
    print(f"\nDoc {i+1}: '{doc}'")
    top_words = list(tfidf_sorted.items())[:5]
    for word, score in top_words:
        print(f"  {word:20} TF-IDF = {score}")

In [ ]:
# Using sklearn TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f'Doc{i+1}' for i in range(len(corpus))]
)

print("\nTF-IDF Matrix (sklearn):")
tfidf_df.round(4)